# PROSIT — C'est quoi le problème ?

## 1. Contexte

Dans la continuité des deux prosits précédents, on ne cherche plus seulement à modéliser le problème de tournée, mais à proposer une méthode de résolution exploitable. Le contexte reste celui des tournées de livraison étudiées comme un **TSP métrique** : il faut visiter chaque ville en minimisant le coût total, tout en sachant désormais que ce problème est difficile à résoudre exactement à grande échelle. Agathe recommande donc d’étudier d’abord une modélisation en Programmation Linéaire en Nombres Entiers, puis d’analyser les limites de cette approche ainsi que celles de la programmation dynamique, avant d’envisager d’autres méthodes.


## 2. Mots inconnus

**Programmation linéaire** : méthode d’optimisation mathématique qui consiste à minimiser ou maximiser une fonction objectif sous des contraintes linéaires.

**PLNE** : Programmation Linéaire en Nombres Entiers ; méthode consistant à modéliser un problème avec une fonction objectif, des contraintes linéaires et des variables entières.

**Fonction objectif** : expression mathématique que l’on cherche à minimiser ou maximiser, par exemple un coût, un temps ou une distance.

**Solveur** : logiciel capable de résoudre automatiquement un modèle mathématique, par exemple CBC, GLPK, Gurobi ou CPLEX.

**Simplexe** : algorithme classique utilisé pour résoudre des problèmes de programmation linéaire continue.

**Programmation dynamique** : approche algorithmique qui résout un problème en le découpant en sous-problèmes mémorisés pour éviter des recalculs inutiles.

**Held-Karp** : algorithme exact de programmation dynamique pour le TSP, plus efficace que la force brute mais restant exponentiel.

**Heuristique** : méthode pratique qui cherche rapidement une bonne solution sans garantir qu’elle soit optimale.

**Méta-heuristique** : stratégie générale d’exploration qui guide des heuristiques pour trouver de bonnes solutions à des problèmes complexes.

**Solution sous-optimale** : solution non garantie optimale mais suffisamment bonne et obtenue dans un temps acceptable.


## 3. Problématique

Quelle stratégie de résolution faut-il retenir pour le problème de tournée de livraison modélisé comme un TSP métrique, sachant qu’une méthode exacte optimale ne passera pas à l’échelle dès que le nombre de villes devient important ?


## 4. Plan d’action

### 4.1 Rappeler le problème et le critère de décision - 1 h 30

Repartir du TSP métrique déjà étudié, rappeler les villes, les distances, les contraintes de tournée et le critère d’optimisation afin de comparer ensuite les méthodes de résolution.

### 4.2 Construire un premier modèle PLNE - 2 h

Traduire le problème de tournée en fonction objectif et en contraintes linéaires, puis préparer un format compatible avec un solveur.

### 4.3 Illustrer la programmation dynamique sur un cas simple - 2 h

Présenter le principe de la programmation dynamique, notamment en lien avec Held-Karp pour le TSP, puis l’illustrer sur un exemple simple afin de comparer cette logique avec la PLNE.

### 4.4 Étudier les limites de la PLNE et de la programmation dynamique - 1 h 30

Comparer les deux approches selon la taille des instances, la consommation mémoire, le temps de calcul et leur capacité à traiter des tournées avec de nombreuses villes.

### 4.5 Choisir la stratégie retenue pour le projet - 1 h

Conclure sur le choix retenu pour le TSP : utiliser une approche heuristique comme méthode principale de production, et réserver la PLNE ou la programmation dynamique aux petites instances et à la validation.


## 5. Réalisation

### 5.1 Rappeler le problème et le critère de décision

Dans la continuité du PROSIT 2, le problème métier correspond à une **tournée de livraison** modélisée comme un **TSP métrique** : il faut visiter chaque ville une seule fois et revenir au point de départ en minimisant le coût total. L’objectif ici n’est donc plus de redéfinir tout le problème, mais de choisir une méthode de résolution adaptée à ce cadre.

Pour comparer les approches de résolution, il faut rappeler les éléments suivants :

- ensemble des villes et matrice des distances ;
- variables de décision ;
- contraintes de visite et de tournée ;
- critère d’optimisation ;
- taille attendue des instances.

Exemple de formalisation simplifiée pour le TSP :

- ensemble $V$ : villes à visiter ;
- distance $d_{ij}$ : coût du trajet de la ville $i$ vers la ville $j$ ;
- variable binaire $x_{ij} \in \{0,1\}$ : 1 si la tournée va directement de $i$ vers $j$, 0 sinon.

On cherche alors à minimiser le coût total de la tournée :

$$
\min \sum_{i \in V} \sum_{j \in V, j \neq i} d_{ij} x_{ij}
$$

sous :

$$
\sum_{j \in V, j \neq i} x_{ij} = 1 \quad \forall i \in V
$$

$$
\sum_{i \in V, i \neq j} x_{ij} = 1 \quad \forall j \in V
$$

$$
x_{ij} \in \{0,1\}
$$

Ces contraintes garantissent qu’une ville est quittée une seule fois et rejointe une seule fois. Dans un vrai modèle TSP, il faut ensuite ajouter des contraintes supplémentaires pour empêcher la formation de plusieurs petits cycles. Cette formalisation montre néanmoins comment le problème étudié dans les prosits précédents peut être traduit en PLNE.


In [1]:
instance = {
    "items": [
        {"id": 1, "cost": 4, "value": 10},
        {"id": 2, "cost": 2, "value": 4},
        {"id": 3, "cost": 6, "value": 12}
    ],
    "capacity": 8
}

**Validation attendue**

- vérification que chaque contrainte métier a une traduction mathématique ;
- revue croisée de la formalisation par un autre membre du groupe ;
- jeu de cas simple calculable à la main pour vérifier le sens du modèle.


### 5.2 Construire un premier modèle PLNE

Une fois le problème formalisé, la mise en œuvre la plus concrète consiste à produire un modèle PLNE exécutable avec un solveur réel. Pour une première mise en œuvre, un bon choix est d’utiliser **Python + PuLP** ou **Python + OR-Tools**. PuLP est un bon point de départ car sa syntaxe est lisible et reste indépendante du solveur utilisé.

Dans le problème réel issu du PROSIT 2, on manipulerait des variables de type $x_{ij}$ pour représenter les trajets entre villes. L’exemple ci-dessous reprend toutefois un cas simple de type **sac à dos**. Le but n’est pas de coder un solveur nous-mêmes, mais de montrer comment une formulation mathématique se traduit en code : les données d’entrée deviennent des structures Python, les variables de décision deviennent des variables binaires, la fonction objectif devient une somme à optimiser et la contrainte devient une équation ou une inéquation.


In [17]:
# !pip install pulp
import pulp

# Donnees d'entree : chaque objet possede un cout et une valeur.
items = [
    {"id": 1, "cost": 4, "value": 10},
    {"id": 2, "cost": 2, "value": 4},
    {"id": 3, "cost": 6, "value": 12}
]

# Capacite maximale du sac.
capacity = 8

# Creation d'un probleme d'optimisation en maximisation.
model = pulp.LpProblem("Optimisation", pulp.LpMaximize)

# Variable binaire pour chaque objet : 1 si l'objet est choisi, 0 sinon.
x = {
    item["id"]: pulp.LpVariable(f"x_{item['id']}", cat="Binary")
    for item in items
}

# Fonction objectif : maximiser la valeur totale des objets retenus.
model += pulp.lpSum(item["value"] * x[item["id"]] for item in items), "Objective"

# Contrainte : le cout total ne doit pas depasser la capacite.
model += pulp.lpSum(item["cost"] * x[item["id"]] for item in items) <= capacity, "Capacity"

# Lancement de la resolution avec le solveur CBC sans afficher les logs techniques.
model.solve(pulp.PULP_CBC_CMD(msg=False))

# Recuperation des objets retenus dans la solution finale.
selected_items = [item for item in items if x[item["id"]].value() == 1]
total_cost = sum(item["cost"] for item in selected_items)
total_value = sum(item["value"] for item in selected_items)

# Affichage d'un resume lisible du resultat.
print("Statut :", pulp.LpStatus[model.status])
print("Valeur optimale :", int(pulp.value(model.objective)))
print("Objets retenus :", [item["id"] for item in selected_items])
print("Cout total :", total_cost)
print("Valeur totale :", total_value)


Statut : Optimal
Valeur optimale : 16
Objets retenus : [2, 3]
Cout total : 8
Valeur totale : 16


Comment lire ce code :

- `items` et `capacity` representent les donnees d’entree du probleme ;
- `LpProblem(..., LpMaximize)` indique que l’on cherche a maximiser la fonction objectif ;
- `LpVariable(..., cat="Binary")` cree les variables de decision binaires du modele ;
- `lpSum(...)` correspond aux sommes mathematiques de la formulation PLNE ;
- `model += ...` sert a ajouter la fonction objectif puis les contraintes ;
- `model.solve(...)` lance la resolution ;
- `x[i].value()` permet de savoir si un objet est retenu ou non dans la solution finale.


Cet exemple reste volontairement simple : il ne cherche pas a representer directement tout le TSP et ses contraintes de tournée, mais a montrer clairement comment une PLNE peut etre codee et resolue avec un outil standard.

Ce choix permet aussi de travailler dans un environnement reproductible et proche d’un contexte de projet réel.



### 5.3 Illustrer la programmation dynamique sur un cas simple

Pour compléter la PLNE, il faut rappeler que, dans le cas du TSP, la programmation dynamique renvoie notamment à l’algorithme de **Held-Karp**, déjà mentionné dans la réflexion sur les méthodes exactes. Son principe est de mémoriser les meilleurs coûts obtenus pour des sous-ensembles de villes. Ici, pour garder un exemple court et lisible, on illustre le mécanisme général de la programmation dynamique sur un cas simple de sac à dos.


In [ ]:
def knapsack_dp(items, capacity):
    n = len(items)

    # dp[i][w] represente la meilleure valeur obtenue avec les i premiers objets
    # et une capacite maximale w.
    dp = [[0] * (capacity + 1) for _ in range(n + 1)]

    for i in range(1, n + 1):
        cost = items[i - 1]["cost"]
        value = items[i - 1]["value"]

        for w in range(capacity + 1):
            # Cas 1 : on ne prend pas l'objet courant.
            dp[i][w] = dp[i - 1][w]

            # Cas 2 : on le prend s'il tient dans la capacite restante.
            if cost <= w:
                dp[i][w] = max(dp[i][w], dp[i - 1][w - cost] + value)

    return dp[n][capacity]


In [ ]:
dp_value = knapsack_dp(items, capacity)
print("Valeur optimale par programmation dynamique :", dp_value)


Pourquoi cette étape est utile :

- elle montre concrètement le principe de la programmation dynamique ;
- elle aide à comprendre la logique de Held-Karp pour le TSP, même si l’exemple ici est plus simple ;
- elle met déjà en évidence que cette approche dépend fortement de la taille de l’espace d’états.


In [ ]:
plne_value = int(pulp.value(model.objective))
print({
    "PLNE": plne_value,
    "programmation_dynamique": dp_value
})


### 5.4 Étudier les limites de la PLNE et de la programmation dynamique

Pour la **PLNE**, les limites apparaissent surtout lorsque le nombre de variables, de contraintes et de relations entre villes devient trop important.

Pour la **programmation dynamique**, les limites apparaissent lorsque l’espace d’états devient trop grand ou trop coûteux à stocker. Dans le cas du TSP, c’est précisément ce qui rend Held-Karp exact mais encore exponentiel.

Dans les deux cas, on obtient des approches exactes intéressantes pour comprendre le problème et traiter de petites tournées, mais elles peuvent devenir trop coûteuses dès que le nombre de villes augmente.

En s’appuyant sur l’analyse du PROSIT 2, on peut retenir un ordre de grandeur simple :

- jusqu’à environ **20 villes**, une méthode exacte reste envisageable pour des tests ou des validations ;
- au-delà, le coût de calcul devient rapidement trop élevé pour un usage courant ;
- pour des tournées avec **plusieurs dizaines ou centaines de villes**, il faut privilégier une méthode heuristique.

Ces seuils restent indicatifs : ils dépendent de l’implémentation, du solveur, de la machine utilisée et de la structure de l’instance.


In [ ]:
import time

start = time.perf_counter()
model.solve(pulp.PULP_CBC_CMD(msg=False))
plne_value = int(pulp.value(model.objective))
plne_duration = time.perf_counter() - start

start = time.perf_counter()
dp_value = knapsack_dp(items, capacity)
dp_duration = time.perf_counter() - start

print({
    "PLNE": {"value": plne_value, "duration_s": round(plne_duration, 6)},
    "programmation_dynamique": {"value": dp_value, "duration_s": round(dp_duration, 6)}
})


Comparaison réaliste :

- **PLNE** : très bonne pour modéliser les contraintes du TSP, utile surtout sur de petites instances ou pour valider un modèle ;
- **programmation dynamique** : pertinente pour comprendre une méthode exacte comme Held-Karp, mais limitée par l’explosion de l’espace d’états ;
- **heuristiques / méta-heuristiques** : les seules approches réellement compatibles avec une utilisation rapide sur des tournées de plusieurs dizaines ou centaines de villes.


### 5.5 Choix retenu pour le TSP

Le choix retenu pour le projet est une **approche heuristique**. Ce choix est cohérent avec ce qui a été montré dans le PROSIT 2 : le TSP est trop coûteux à résoudre exactement dès que le nombre de villes augmente. La **PLNE** et la **programmation dynamique** restent utiles pour comprendre le problème, construire un modèle rigoureux et valider de petites instances, mais elles ne constituent pas la solution principale à retenir pour une exploitation à grande échelle.

En pratique, on peut formuler la règle de décision ainsi :

- pour **de petites instances** (de l’ordre de **20 villes ou moins**), une méthode exacte peut encore servir à vérifier ou comparer les solutions ;
- pour **des instances réelles** avec **plusieurs dizaines ou centaines de villes**, on retient une **heuristique** comme méthode principale ;
- si la qualité d’une heuristique simple n’est pas suffisante, on pourra la renforcer par une **méta-heuristique** ou une amélioration locale.


In [ ]:
cities = ["A", "B", "C", "D"]
distances = {
    ("A", "B"): 10, ("A", "C"): 15, ("A", "D"): 20,
    ("B", "C"): 35, ("B", "D"): 25,
    ("C", "D"): 30,
}

def dist(u, v):
    if u == v:
        return 0
    return distances.get((u, v), distances[(v, u)])

def nearest_neighbor_tsp(cities, start):
    unvisited = set(cities)
    unvisited.remove(start)
    route = [start]
    total_cost = 0
    current = start

    while unvisited:
        next_city = min(unvisited, key=lambda city: dist(current, city))
        total_cost += dist(current, next_city)
        route.append(next_city)
        unvisited.remove(next_city)
        current = next_city

    total_cost += dist(current, start)
    route.append(start)

    return {"route": route, "cost": total_cost}

nearest_neighbor_tsp(cities, start="A")


**Décision retenue**

- la **stratégie retenue** pour le projet est une **heuristique** de résolution du TSP ;
- la **PLNE** sert de référence pour modéliser rigoureusement le problème et tester des instances d’environ **20 villes ou moins** ;
- la **programmation dynamique** sert surtout à comprendre une autre approche exacte, mais ne constitue pas le choix principal au-delà des petites tailles ;
- une **méta-heuristique** peut être envisagée ensuite si une heuristique simple ne donne pas une qualité suffisante.

En conclusion, le problème étant déjà identifié comme un TSP métrique difficile, la bonne démarche n’est pas de retenir une méthode exacte comme solution principale. La décision la plus cohérente est d’utiliser une **heuristique** pour produire rapidement une tournée exploitable sur des instances réelles, et de réserver la **PLNE** ou la **programmation dynamique** aux petites tailles, typiquement autour de **20 villes ou moins**, pour l’analyse et la validation.
